# Python程序运行时间优化方法

## 1. **向量化计算（NumPy/Pandas）**

### NumPy向量化操作

In [ ]:
import numpy as np

# 慢：Python循环
result = []
for i in range(len(a)):
    result.append(a[i] + b[i])

# 快：NumPy向量化
a_np = np.array(a)
b_np = np.array(b)
result = a_np + b_np  # 快10-100倍

# 更复杂的向量化运算
result = np.sqrt(a_np**2 + b_np**2)
result = np.exp(a_np) * np.sin(b_np)


### Pandas向量化操作

In [ ]:
import pandas as pd

# 慢：逐行处理
df['new_col'] = 0
for i in range(len(df)):
    df.loc[i, 'new_col'] = df.loc[i, 'col1'] + df.loc[i, 'col2']

# 快：向量化操作
df['new_col'] = df['col1'] + df['col2']

# 使用apply也比循环快
df['new_col'] = df.apply(lambda row: row['col1'] * row['col2'], axis=1)

# 条件向量化操作
df['category'] = np.where(df['value'] > 100, 'high', 'low')


## 2. **JIT编译（Numba）**

In [ ]:
from numba import jit
import numpy as np

# 普通Python函数（慢）
def slow_sum(arr):
    total = 0
    for i in range(len(arr)):
        total += arr[i]
    return total

# JIT编译版本（快）
@jit(nopython=True)
def fast_sum(arr):
    total = 0
    for i in range(len(arr)):
        total += arr[i]
    return total

# 使用
arr = np.random.rand(1000000)
result = fast_sum(arr)  # 第一次运行会编译，之后极快


## 3. **使用高效的数据结构**

### 集合和字典查找

In [ ]:
# 慢：列表查找 O(n)
if item in my_list:  # 每次都要扫描整个列表
    pass

# 快：集合查找 O(1)
my_set = set(my_list)
if item in my_set:   # 哈希查找，极快
    pass

# 字典用于快速映射
lookup_dict = {key: value for key, value in zip(keys, values)}
result = lookup_dict.get(some_key, default_value)


### 使用deque进行队列操作

In [ ]:
from collections import deque

# 列表作为队列（慢）
queue = []
queue.append(item)        # O(1)
item = queue.pop(0)       # O(n) - 需要移动所有元素

# deque作为队列（快）
queue = deque()
queue.append(item)        # O(1)
item = queue.popleft()    # O(1)


## 4. **算法优化**

### 选择合适的数据结构

In [ ]:
# 需要频繁插入删除：链表、deque
# 需要快速查找：集合、字典
# 需要排序：堆、有序字典
# 需要范围查询：平衡树（bisect模块）


### 减少时间复杂度

In [ ]:
import bisect

# O(n²) -> O(n log n)
def find_pairs_optimized(arr, target):
    arr.sort()  # O(n log n)
    pairs = []
    left, right = 0, len(arr) - 1
    while left < right:  # O(n)
        current_sum = arr[left] + arr[right]
        if current_sum == target:
            pairs.append((arr[left], arr[right]))
            left += 1
            right -= 1
        elif current_sum < target:
            left += 1
        else:
            right -= 1
    return pairs


## 5. **内置函数和库函数**

### 使用内置函数

In [ ]:
# 慢：手动实现
total = 0
for num in numbers:
    total += num

# 快：内置函数
total = sum(numbers)

# 其他高效内置函数
min_val = min(numbers)
max_val = max(numbers)
sorted_nums = sorted(numbers)


### 使用itertools

In [ ]:
import itertools

# 高效的迭代器操作
# 组合
for combo in itertools.combinations(items, 2):
    process(combo)

# 排列
for perm in itertools.permutations(items, 2):
    process(perm)

# 分组
for key, group in itertools.groupby(data, key_func):
    process_group(list(group))


## 6. **并行计算**

### 多进程（CPU密集型）

In [ ]:
from multiprocessing import Pool
import numpy as np

def process_chunk(chunk):
    return np.sum(chunk ** 2)

def parallel_processing(data, num_processes=4):
    chunk_size = len(data) // num_processes
    chunks = [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]
    
    with Pool(num_processes) as pool:
        results = pool.map(process_chunk, chunks)
    
    return sum(results)

# 使用
data = np.random.rand(1000000)
result = parallel_processing(data)


### 多线程（I/O密集型）

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import requests

def fetch_url(url):
    return requests.get(url).text

urls = ['http://example.com/1', 'http://example.com/2', ...]

with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(fetch_url, urls))


## 7. **Pandas高级优化技巧**

### 使用合适的数据类型

In [ ]:
# 优化数据类型减少内存和计算时间
df['int_col'] = df['int_col'].astype('int32')
df['float_col'] = df['float_col'].astype('float32')
df['category_col'] = df['category_col'].astype('category')

# 使用query进行快速过滤
result = df.query('col1 > 100 & col2 == "value"')

# 使用eval进行表达式计算（大数据集）
df.eval('new_col = col1 + col2 * col3', inplace=True)


### 批量操作替代循环

In [ ]:
# 慢：逐行操作
for idx, row in df.iterrows():
    df.loc[idx, 'new_col'] = complex_calculation(row)

# 快：向量化操作或apply
df['new_col'] = df.apply(complex_calculation, axis=1)

# 更快：完全向量化（如果可能）
df['new_col'] = df['col1'] * 0.8 + df['col2'] * 0.2


## 8. **缓存和记忆化**

In [ ]:
from functools import lru_cache
import time

# 缓存昂贵函数的结果
@lru_cache(maxsize=128)
def expensive_function(x, y):
    time.sleep(1)  # 模拟耗时操作
    return x * y + x - y

# 第一次调用会计算，后续相同参数直接返回结果
result1 = expensive_function(10, 20)  # 耗时1秒
result2 = expensive_function(10, 20)  # 立即返回


## 9. **字符串操作优化**

In [ ]:
# 慢：字符串拼接
result = ""
for s in string_list:
    result += s

# 快：join操作
result = "".join(string_list)

# 使用字符串方法而非正则表达式（简单情况）
# 慢：re.match(pattern, text)
# 快：text.startswith(prefix)


## 10. **性能分析工具**

### 使用cProfile分析性能

In [ ]:
import cProfile

def your_function():
    # 你的代码
    pass

# 分析性能
cProfile.run('your_function()')


### 使用line_profiler逐行分析

In [ ]:
# 安装: pip install line_profiler
# 使用: kernprof -l -v script.py

@profile
def slow_function():
    # 需要分析的函数
    result = 0
    for i in range(1000000):
        result += i
    return result


## 11. **编译优化**

### 使用Cython

In [ ]:
# 文件: fast_module.pyx
def compute(int n):
    cdef int i, result = 0
    for i in range(n):
        result += i * i
    return result

# 编译后比纯Python快很多


### 使用PyPy解释器
- PyPy具有JIT编译器，对纯Python代码有很好的加速效果
- 特别适合长时间运行的应用程序

## 12. **数据库和文件I/O优化**

In [ ]:
# 批量数据库操作
# 慢：逐条插入
for item in data:
    cursor.execute("INSERT INTO table VALUES (?, ?)", (item[0], item[1]))

# 快：批量插入
cursor.executemany("INSERT INTO table VALUES (?, ?)", data)

# 文件读取优化
import pandas as pd
# 指定数据类型和只读需要的列
df = pd.read_csv('large_file.csv', dtype={'col1': 'int32'}, usecols=['col1', 'col2'])


## 实际应用示例

In [ ]:
import numpy as np
import pandas as pd
from numba import jit

# 优化前的慢版本
def slow_data_processing(data):
    results = []
    for row in data:
        if row[0] > 100:
            results.append(row[1] * row[2] + np.sqrt(row[3]))
        else:
            results.append(0)
    return results

# 优化后的快版本
@jit(nopython=True)
def fast_data_processing_numpy(data):
    # data是numpy数组
    results = np.zeros(len(data))
    for i in range(len(data)):
        if data[i, 0] > 100:
            results[i] = data[i, 1] * data[i, 2] + np.sqrt(data[i, 3])
    return results

# 或者使用纯Pandas向量化
def fast_data_processing_pandas(df):
    mask = df['col0'] > 100
    df['result'] = 0
    df.loc[mask, 'result'] = df['col1'] * df['col2'] + np.sqrt(df['col3'])
    return df['result'].values


这些优化方法可以根据具体场景组合使用，通常能获得数倍到数百倍的性能提升。关键是先分析瓶颈所在，然后针对性地应用合适的优化技术。